# REVE 模型训练 - 运动意图四分类

本 notebook 将 REVE 预训练模型应用到自定义 EEG 数据集，进行运动意图四分类任务。

**数据要求：**
- 64 通道 EEG，标准 10-10 系统
- 原始采样率 250 Hz，需重采样至 200 Hz（REVE 要求）
- 四分类任务（运动意图）

**REVE 相关说明：**
- `pos` 张量：形状 `(batch_size, 64, 3)`，表示每个通道的 3D 坐标 (x,y,z)
- 使用 `reve-positions` 的 `pos_bank(ch_names)` 自动获取标准 10-10 坐标（即 get_positions 功能）
- 预训练模型与电极位置库已改为从本地 `./REVE` 加载；若需 `reve-positions` 离线，请下载到 `./REVE_positions`

**训练策略：** 冻结前 12 层、数据增强、Mixup、强 dropout、分组学习率、warmup、早停。

## 依赖安装（如未安装）

```bash
pip install torch transformers scipy scikit-learn einops
# 可选（加速）：pip install flash-attn
```

In [7]:
# ========== 1. 环境与依赖 ==========
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from scipy.signal import resample
import sys
import os

# 添加 REVE 路径，以便 transformers 加载 trust_remote_code 时找到 modeling_reve、configuration_reve
REVE_PATH = os.path.abspath("./REVE")  # REVE 文件夹路径（含 configuration_reve.py, modeling_reve.py）
if REVE_PATH not in sys.path:
    sys.path.insert(0, REVE_PATH)

# 数据路径占位符
DATA_PATH = "./Dataset/eeg_epochs_4class_0to6s.npz"  # 预处理后的 npz 文件路径
PRETRAINED_MODEL_PATH = "./REVE"  # 本地 REVE 预训练模型路径（含 config.json, model.safetensors）
_pos_local = os.path.join(os.path.dirname(REVE_PATH), "REVE_positions")  # 与 REVE 同级目录
POS_BANK_PATH = _pos_local if os.path.isdir(_pos_local) else "brain-bzh/reve-positions"  # 优先本地，否则从 HF 加载

# 参数定义
TARGET_SFREQ = 200  # REVE 要求输入采样率为 200 Hz
N_CLASSES = 4  # 四分类
BATCH_SIZE = 16
EPOCHS = 100
LR_BACKBONE = 2e-5  # backbone 更小学习率，减轻过拟合
LR_HEAD = 2e-4  # 分类头学习率
DROPOUT = 0.5  # 更强 dropout
PATIENCE = 15  # 早停
FREEZE_LAYERS = 12  # 冻结前 12 层 transformer，只微调后半部分
# 数据增强
AUG_TIME_SHIFT = 50  # 时间平移最大点数（约 0.25s @ 200Hz）
AUG_NOISE_STD = 0.02  # 高斯噪声标准差（相对信号幅度）
AUG_AMP_RANGE = (0.9, 1.1)  # 幅度缩放范围
AUG_CH_DROP = 0.05  # 通道 dropout 概率
MIXUP_ALPHA = 0.2  # Mixup alpha，0 表示关闭
WARMUP_EPOCHS = 3  # 学习率 warmup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
# ========== 2. 数据加载与重采样 ==========
# 从 data_processing 导出的 npz 加载数据
# 原始采样率 250 Hz，必须重采样到 200 Hz（REVE 要求）
# 可选：在 data_processing 中 epochs 提取后对 raw 使用 raw.resample(200)，再保存 npz，则此处可跳过重采样

def load_and_resample_data(data_path, target_sfreq=200):
    """
    加载 npz 数据并将 EEG 从原始采样率重采样到 target_sfreq。
    使用 scipy.signal.resample 进行重采样。
    """
    d = np.load(data_path, allow_pickle=True)
    X = d["X"]  # (N, C, T)
    y = d["y"]
    sfreq = float(d["sfreq"])
    ch_names = d["ch_names"]
    
    if sfreq != target_sfreq:
        # 重采样：新长度 = 原长度 * target_sfreq / sfreq
        n_times_orig = X.shape[2]
        n_times_new = int(n_times_orig * target_sfreq / sfreq)
        X_resampled = np.zeros((X.shape[0], X.shape[1], n_times_new), dtype=np.float32)
        for i in range(X.shape[0]):
            for c in range(X.shape[1]):
                X_resampled[i, c, :] = resample(X[i, c, :], n_times_new)
        X = X_resampled
        print(f"重采样: {sfreq} Hz -> {target_sfreq} Hz, 时间点数: {n_times_orig} -> {n_times_new}")
    
    return X, y, ch_names, target_sfreq

X, y, ch_names, sfreq = load_and_resample_data(DATA_PATH, TARGET_SFREQ)
ch_names = list(ch_names)  # 转为 list，供 pos_bank 使用

print("X:", X.shape, X.dtype)
print("y:", y.shape, "classes:", np.unique(y))
print("sfreq:", sfreq)
print("ch_names[:5]:", ch_names[:5])

重采样: 250.0 Hz -> 200 Hz, 时间点数: 1501 -> 1200
X: (1385, 64, 1200) float32
y: (1385,) classes: [0 1 2 3]
sfreq: 200
ch_names[:5]: [np.str_('FP1'), np.str_('FPz'), np.str_('FP2'), np.str_('AF7'), np.str_('AF3')]


In [9]:
# ========== 3. 划分训练/验证/测试集 ==========
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=42, shuffle=True, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, shuffle=True, stratify=y_tmp
)

print("train/val/test:", len(y_train), len(y_val), len(y_test))

train/val/test: 969 208 208


In [10]:
# ========== 4. REVE 电极位置（pos）与 Dataset ==========
# REVE 使用 pos 张量表示每个通道的 3D 空间坐标 (x, y, z)
# 形状必须为 (batch_size, n_channels, 3)
# 使用 reve-positions 的 pos_bank(electrode_names) 自动获取标准 10-10 坐标

from transformers import AutoModel

# 加载电极位置库（get_positions 功能通过 pos_bank(ch_names) 实现）
pos_bank = AutoModel.from_pretrained(POS_BANK_PATH, trust_remote_code=True)

# 通道名映射：position bank 对大小写敏感，部分命名与 MNE/数据不同，需统一
# 例如 position bank 有 Fpz/FPZ 但无 FPz，数据中 FPz 需映射为 Fpz
CH_NAME_MAP = {"FPz": "Fpz", "FP1": "Fp1", "FP2": "Fp2"}  # 数据名 -> position bank 名
ch_names_for_pos = [CH_NAME_MAP.get(str(c), str(c)) for c in ch_names]

# 获取 64 通道的 3D 坐标，形状 (64, 3)
pos_single = pos_bank(ch_names_for_pos)  # 等价于 get_positions：根据通道名获取坐标
pos_single = pos_single.detach()  # 转为普通 tensor，便于后续 expand

print("pos_single shape:", pos_single.shape)  # 应为 (64, 3)
if pos_single.shape[0] != 64:
    # 找出 position bank 中不存在的通道名
    all_pos = set(pos_bank.get_all_positions())
    missing = [c for c in ch_names_for_pos if c not in all_pos]
    raise ValueError(f"有 {len(missing)} 个通道在 position bank 中无对应: {missing}。请在 CH_NAME_MAP 中添加映射。")
assert pos_single.shape == (64, 3), f"pos 应为 (64, 3)，当前为 {pos_single.shape}"


def augment_eeg(eeg, time_shift, noise_std, amp_range, ch_drop, rng):
    """EEG 数据增强：时间平移、高斯噪声、幅度缩放、通道 dropout"""
    eeg = eeg.clone()
    C, T = eeg.shape
    if time_shift > 0:
        shift = int(rng.integers(-time_shift, time_shift + 1))
        if shift != 0:
            eeg = torch.roll(eeg, shift, dims=1)
    if noise_std > 0:
        std = eeg.std().item() * noise_std
        if std > 1e-8:
            eeg = eeg + torch.from_numpy(rng.normal(0, std, eeg.shape).astype(np.float32))
    if amp_range[1] > amp_range[0]:
        scale = float(rng.uniform(amp_range[0], amp_range[1]))
        eeg = eeg * scale
    if ch_drop > 0:
        mask = rng.random(C) > ch_drop
        eeg = eeg * torch.from_numpy(mask.astype(np.float32)).unsqueeze(1)
    return eeg


class REVEDataset(Dataset):
    """
    返回 (eeg, pos, y)。
    train=True 时启用数据增强。
    """
    def __init__(self, X, y, pos_template, train=False):
        self.X = torch.from_numpy(X).float()  # (N, C, T)
        self.y = torch.from_numpy(y).long()
        self.pos = pos_template  # (C, 3)
        self.train = train
        self.rng = np.random.default_rng()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        eeg = self.X[i].clone()  # (C, T)
        if self.train:
            eeg = augment_eeg(
                eeg, AUG_TIME_SHIFT, AUG_NOISE_STD, AUG_AMP_RANGE, AUG_CH_DROP,
                self.rng
            )
        pos = self.pos  # (C, 3)
        return eeg, pos, self.y[i]


train_ds = REVEDataset(X_train, y_train, pos_single, train=True)
val_ds = REVEDataset(X_val, y_val, pos_single, train=False)
test_ds = REVEDataset(X_test, y_test, pos_single, train=False)

def collate_fn(batch):
    """将 pos 从 (C, 3) 扩展为 (B, C, 3)"""
    eegs = torch.stack([b[0] for b in batch])
    pos = batch[0][1]  # 所有样本 pos 相同
    pos = pos.unsqueeze(0).expand(len(batch), -1, -1)  # (B, C, 3)
    ys = torch.stack([b[2] for b in batch])
    return eegs, pos, ys

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

eb, pb, yb = next(iter(train_loader))
print("batch eeg:", eb.shape, "batch pos:", pb.shape, "batch y:", yb.shape)

Loading weights:   0%|          | 0/1 [00:00<?, ?it/s]

pos_single shape: torch.Size([64, 3])
batch eeg: torch.Size([16, 64, 1200]) batch pos: torch.Size([16, 64, 3]) batch y: torch.Size([16])


In [11]:
# ========== 5. REVE 模型与分类头 ==========
# REVE 为 encoder，无分类头。需添加 attention_pooling + Linear 做四分类

from transformers import AutoModel, AutoConfig


class REVEClassifier(nn.Module):
    """
    REVE backbone + 分类头。
    - eeg: (B, C, T)
    - pos: (B, C, 3) 传入 REVE 的 4D 位置编码
    """
    def __init__(self, reve_model, n_classes=4, dropout=0.3):
        super().__init__()
        self.reve = reve_model
        embed_dim = reve_model.embed_dim
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(embed_dim, n_classes)

    def forward(self, eeg, pos):
        # REVE forward，return_output=True 得到 transformer 输出
        x = self.reve(eeg, pos, return_output=True)
        # x: list of tensors，取最后一层并 reshape 为 (B, C, S, E)
        if isinstance(x, list):
            x = x[-1]
        # 根据 modeling_reve：return_output 时 x 为 (B, C*H, E)，需 reshape
        # 实际 return_output 返回的是 transformer 的序列输出，需用 attention_pooling
        # 查看 modeling：return_output 时直接返回 transformer(x)，即 (B, seq, E)
        # 需要 reshape 回 (B, C, H, E) 才能用 attention_pooling
        b, c, t = eeg.shape
        patch_size = self.reve.patch_size
        overlap = self.reve.overlap_size
        step = patch_size - overlap
        n_patches = (t - patch_size) // step + 1
        # x: (B, C*n_patches, E) -> (B, C, n_patches, E)
        x = x.view(b, c, n_patches, -1)
        pooled = self.reve.attention_pooling(x)  # (B, E)
        pooled = self.dropout(pooled)
        return self.classifier(pooled)  # (B, n_classes)


# 加载 REVE 预训练模型
reve_config = AutoConfig.from_pretrained(PRETRAINED_MODEL_PATH, trust_remote_code=True)
reve_model = AutoModel.from_pretrained(PRETRAINED_MODEL_PATH, config=reve_config, trust_remote_code=True)

# 若使用本地 REVE 文件夹（含 config.json）：
# reve_model = AutoModel.from_pretrained("./REVE", trust_remote_code=True)

model = REVEClassifier(reve_model, n_classes=N_CLASSES, dropout=DROPOUT).to(DEVICE)

# 冻结前 N 层 transformer，减轻过拟合（小样本时推荐）
if FREEZE_LAYERS > 0:
    layers = model.reve.transformer.layers
    for i, layer in enumerate(layers):
        if i < FREEZE_LAYERS:
            for p in layer.parameters():
                p.requires_grad = False
    n_frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"冻结前 {FREEZE_LAYERS} 层 | 可训练: {n_total - n_frozen:,} / {n_total:,}")

# 加载 state_dict（若已有微调后的 checkpoint，可选择性加载）
# 示例：加载预训练权重到 backbone，忽略 classifier（因类别数不同）
# ckpt = torch.load("path/to/checkpoint.pt", map_location=DEVICE)
# model.load_state_dict(ckpt["model"], strict=False)  # strict=False 忽略不匹配的 key

model.eval()
with torch.no_grad():
    logits = model(eb.to(DEVICE), pb.to(DEVICE))
print("logits shape:", logits.shape)

Loading weights:   0%|          | 0/140 [00:00<?, ?it/s]

logits shape: torch.Size([16, 4])


In [12]:
# ========== 6. 训练循环 ==========
def accuracy_from_logits(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

def mixup_data(eeg, y, alpha=0.2):
    """Mixup: 混合两个样本，返回混合后的 eeg 和 (lam, y_a, y_b)"""
    if alpha <= 0:
        return eeg, None
    lam = np.random.beta(alpha, alpha)
    batch_size = eeg.size(0)
    index = torch.randperm(batch_size, device=eeg.device)
    mixed_eeg = lam * eeg + (1 - lam) * eeg[index]
    return mixed_eeg, (lam, y, y[index])

def mixup_criterion(criterion, logits, mix_info, yb):
    if mix_info is None:
        return criterion(logits, yb)
    lam, y_a, y_b = mix_info
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)

# 分组学习率：只传入 requires_grad 的参数
backbone_params = [p for p in model.reve.parameters() if p.requires_grad]
head_params = list(model.dropout.parameters()) + list(model.classifier.parameters())
optimizer = torch.optim.AdamW(
    [{"params": backbone_params, "lr": LR_BACKBONE},
     {"params": head_params, "lr": LR_HEAD}],
    weight_decay=1e-2
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=8
)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def run_one_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_acc, total_n = 0.0, 0.0, 0
    for eeg, pos, yb in loader:
        eeg = eeg.to(DEVICE)
        pos = pos.to(DEVICE)
        yb = yb.to(DEVICE)
        mix_info = None
        if train and MIXUP_ALPHA > 0:
            eeg, mix_info = mixup_data(eeg, yb, MIXUP_ALPHA)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            logits = model(eeg, pos)
            if mix_info is not None:
                loss = mixup_criterion(criterion, logits, mix_info, yb)
            else:
                loss = criterion(logits, yb)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        bs = yb.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy_from_logits(logits, yb) * bs
        total_n += bs
    return total_loss / total_n, total_acc / total_n

train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_acc = -1.0
best_state = None
patience_counter = 0

for ep in range(1, EPOCHS + 1):
    if ep <= WARMUP_EPOCHS:
        warmup_factor = ep / WARMUP_EPOCHS
        optimizer.param_groups[0]["lr"] = LR_BACKBONE * warmup_factor
        optimizer.param_groups[1]["lr"] = LR_HEAD * warmup_factor
    tr_loss, tr_acc = run_one_epoch(train_loader, train=True)
    va_loss, va_acc = run_one_epoch(val_loader, train=False)
    train_losses.append(tr_loss)
    val_losses.append(va_loss)
    train_accs.append(tr_acc)
    val_accs.append(va_acc)
    scheduler.step(va_acc)
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
    print(f"Epoch {ep:02d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {va_loss:.4f} acc {va_acc:.3f}")
    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {ep} (no improvement for {PATIENCE} epochs)")
        break

print("best val acc:", best_val_acc)

Epoch 01 | train loss 1.5711 acc 0.244 | val loss 1.4055 acc 0.250
Epoch 02 | train loss 1.4307 acc 0.292 | val loss 1.3970 acc 0.245
Epoch 03 | train loss 1.3560 acc 0.338 | val loss 1.5095 acc 0.250
Epoch 04 | train loss 1.2178 acc 0.470 | val loss 1.6251 acc 0.284
Epoch 05 | train loss 0.9619 acc 0.658 | val loss 1.8063 acc 0.240
Epoch 06 | train loss 0.7103 acc 0.833 | val loss 2.0626 acc 0.284
Epoch 07 | train loss 0.5823 acc 0.926 | val loss 1.9270 acc 0.250
Epoch 08 | train loss 0.4852 acc 0.970 | val loss 1.9184 acc 0.245
Epoch 09 | train loss 0.4509 acc 0.993 | val loss 1.7580 acc 0.308
Epoch 10 | train loss 0.4366 acc 0.992 | val loss 1.6860 acc 0.308
Epoch 11 | train loss 0.4022 acc 0.999 | val loss 1.6603 acc 0.279
Epoch 12 | train loss 0.4067 acc 0.998 | val loss 1.6958 acc 0.264
Epoch 13 | train loss 0.3887 acc 1.000 | val loss 1.6425 acc 0.298
Epoch 14 | train loss 0.3794 acc 1.000 | val loss 1.7038 acc 0.255
Epoch 15 | train loss 0.3743 acc 1.000 | val loss 1.6197 acc 0

KeyboardInterrupt: 

In [ ]:
# ========== 7. 测试集评估 ==========
model.load_state_dict(best_state)
model.to(DEVICE)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for eeg, pos, yb in test_loader:
        eeg, pos = eeg.to(DEVICE), pos.to(DEVICE)
        logits = model(eeg, pos)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.numpy())

y_pred = np.array(all_preds)
y_true = np.array(all_labels)
test_acc = (y_pred == y_true).mean()
print("Test accuracy:", test_acc)
print("test true counts:", np.bincount(y_true, minlength=N_CLASSES))
print("test pred counts:", np.bincount(y_pred, minlength=N_CLASSES))

In [ ]:
# ========== 8. 保存模型（含 state_dict 加载示例）==========
# 保存完整模型和 state_dict
torch.save({
    "model": model.state_dict(),
    "config": {"n_classes": N_CLASSES, "sfreq": sfreq},
}, "reve_4class_best.pt")

# 加载 state_dict 示例：
# ckpt = torch.load("reve_4class_best.pt", map_location=DEVICE)
# model.load_state_dict(ckpt["model"], strict=True)
# 若只加载 backbone 而 classifier 结构不同，使用 strict=False